# ⚡ CodeXcelerate: Python-to-C++ AI Transpiler
## AlgoProfessor AI R&D Internship — Day 30 | Milestone 5

---

### 🎯 What This Notebook Does

This notebook builds a **complete AI-powered transpiler** that:
1. Takes **real Python code** as input (computationally intensive programs)
2. Uses **GPT-4o in two passes** to generate optimised, parallelised C++ code
3. **Compiles** the C++ with `clang++` using `-O3` and OpenMP flags
4. **Benchmarks** both versions using `hyperfine` (industry-standard benchmarking tool)
5. **Visualises** the speedup with interactive Plotly charts

### 📊 Expected Results
| Algorithm | Python Time | C++ Time | Speedup |
|-----------|-------------|----------|---------|
| Bubble Sort (10k items) | ~800ms | ~0.5ms | ~1,600× |
| Prime Sieve (5M) | ~2,500ms | ~40ms | ~60× |
| Matrix Multiply (500×500) | ~45,000ms | ~8ms | ~5,600× |

### 🛠️ Tech Stack
- **Groq + Llama 3.3 70B** — AI transpilation engine (2-pass: convert → parallelise) — FREE!
- **Clang++** — C++ compiler with `-O3 -fopenmp -march=native`
- **OpenMP** — CPU parallelism library (uses all available cores)
- **Hyperfine** — Statistical command-line benchmarking tool
- **Plotly** — Interactive visualisation

### ⚠️ Prerequisites
- A **FREE Groq API key** → sign up at **console.groq.com** (no credit card!)
- A Google Colab account (free tier is sufficient for all cells)
- 💡 **Groq is 100% free** with 14,400 requests/day limit — more than enough for this project!

---
> **How to use this notebook:** Run cells top to bottom using `Shift+Enter`. Each cell has a markdown explanation above the code. Read the explanation first, then run the code, then check the expected output.

---
## 🔧 Cell 1 — Environment Setup

### What this cell does:
Installs all **system-level tools** that Python's `pip` cannot install — these are Linux packages.

| Package | Purpose |
|---------|----------|
| `clang` | The C++ compiler we use to compile GPT-4o's generated code |
| `libomp-dev` | OpenMP library — enables multi-core CPU parallelism in C++ |
| `hyperfine` | A statistical benchmarking tool that runs programs many times and gives precise timing |

### Why clang++ instead of g++?
Clang produces better error messages (important when debugging LLM-generated code) and is available by default in most cloud environments.

### ⏱️ Expected runtime: 1–3 minutes (downloading packages)
### ✅ Expected output: Version numbers printed for clang++ and hyperfine

In [ ]:
# ─── CELL 1: System Environment Setup ───────────────────────────────────────
# Re-run this cell every time you open a new Colab session.

import subprocess, shutil, glob, os

print('Installing packages...')

# Install clang (whichever version apt provides) + OpenMP
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', 'clang', 'libomp-dev'], capture_output=True)
print('Done: clang + libomp-dev')

# Find the actual clang++ binary — could be clang++-14, clang++-15, clang++-16 etc.
# Search /usr/bin for any clang++ binary
candidates = sorted(glob.glob('/usr/bin/clang++*'))
print(f'Found clang++ binaries: {candidates}')

# Create generic symlink to the first one found
if candidates:
    best = candidates[-1]   # take highest version
    os.makedirs('/usr/local/bin', exist_ok=True)
    subprocess.run(['ln', '-sf', best, '/usr/local/bin/clang++'], capture_output=True)
    print(f'Symlink created: /usr/local/bin/clang++ -> {best}')
else:
    print('WARNING: no clang++ binary found in /usr/bin')

# Install hyperfine
subprocess.run([
    'wget', '-q',
    'https://github.com/sharkdp/hyperfine/releases/download/v1.18.0/hyperfine_1.18.0_amd64.deb'
], capture_output=True)
subprocess.run(['dpkg', '-i', 'hyperfine_1.18.0_amd64.deb'], capture_output=True)
subprocess.run(['rm', '-f', 'hyperfine_1.18.0_amd64.deb'], capture_output=True)
print('Done: hyperfine')

print()
print('-' * 50)
print('Verifying:')

# Verify clang++ — try symlink first, then any versioned binary
clang_ok = False
for candidate in ['/usr/local/bin/clang++'] + sorted(glob.glob('/usr/bin/clang++*'), reverse=True):
    r = subprocess.run([candidate, '--version'], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'OK  clang++ : {r.stdout.splitlines()[0]}')
        print(f'    path    : {candidate}')
        # Store the working path in env so Cell 7 can find it
        os.environ['CLANGPP_PATH'] = candidate
        clang_ok = True
        break
if not clang_ok:
    print('FAIL: no working clang++ found!')
    print('Try running:  !apt-get install -y clang')

# Verify hyperfine
if shutil.which('hyperfine'):
    r = subprocess.run(['hyperfine', '--version'], capture_output=True, text=True)
    print(f'OK  hyperfine: {r.stdout.strip()}')
    hyper_ok = True
else:
    print('FAIL: hyperfine not found')
    hyper_ok = False

# CPU cores
r = subprocess.run(['nproc'], capture_output=True, text=True)
print(f'OK  CPU cores: {r.stdout.strip()}')

print()
print('-' * 50)
if clang_ok and hyper_ok:
    print('All tools ready. Proceed to Cell 2.')
else:
    print('WARNING: re-run this cell before continuing!')


---
## 📚 Cell 2 — Install Python Libraries

### What this cell does:
Installs all Python packages we need using `pip`.

| Library | Purpose |
|---------|----------|
| `openai` | Official OpenAI Python SDK — lets us call GPT-4o |
| `plotly` | Interactive charts for visualising benchmark results |
| `pandas` | Data manipulation — we store benchmark results in a DataFrame |
| `rich` | Beautiful terminal output (progress bars, coloured text) |

### Why these specific versions?
We pin `openai>=1.0` because the API changed significantly in v1.0 — older code will not work.

### ⏱️ Expected runtime: 30–60 seconds
### ✅ Expected output: Library version confirmation

In [ ]:
# ─── CELL 2: Python Library Installation ────────────────────────────────────
# -q flag suppresses verbose pip output so we only see important messages

print('📦 Installing Python libraries...')

!pip install groq plotly pandas rich -q

print()
print('─' * 50)
print('✅ Libraries installed. Verifying imports:')
print()

import groq
import plotly
import pandas
import rich

print(f'   groq    version: {groq.__version__}')
print(f'   plotly  version: {plotly.__version__}')
print(f'   pandas  version: {pandas.__version__}')
from importlib.metadata import version as pkg_version
print(f'   rich    version: {pkg_version("rich")}')
print()
print('─' * 50)
print('✅ All libraries ready. Proceed to Cell 3.')

---
## 📦 Cell 3 — Imports and API Configuration

### What this cell does:
Imports all Python modules and **securely loads your Groq API key** from Colab Secrets.

### 🔑 How to get your FREE Groq API key:
1. Go to **console.groq.com** and sign up (free — no credit card)
2. Click **"API Keys" → "Create API Key"** → copy it
3. In Colab: click the **🔑 key icon** in the left sidebar
4. Click **"Add new secret"** → Name: `GROQ_API_KEY` → paste your key
5. Toggle **"Notebook access"** to ON

### Why Groq instead of OpenAI?
✅ **100% Free** — generous free tier (14,400 requests/day)
✅ **Extremely fast** — ~800 tokens/sec (10× faster than OpenAI)
✅ **Compatible API** — same structure as OpenAI, minimal code changes
✅ **No credit card** required to sign up

### ✅ Expected output: 'API client ready' confirmation

In [ ]:
# ─── CELL 3: Imports and Configuration ──────────────────────────────────────

# Standard library
import os
import json
import subprocess
import tempfile
import time
import textwrap
from pathlib import Path
from datetime import datetime

# Third-party libraries
from groq import Groq
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table
from rich import print as rprint

# ─── Load API Key from Colab Secrets ────────────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('✅ Groq API key loaded from Colab Secrets')
except Exception:
    # Fallback: load from environment variable (for local development)
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')
    if GROQ_API_KEY:
        print('✅ Groq API key loaded from environment variable')
    else:
        print('⚠️  No API key found!')
        print('   Go to console.groq.com → get free API key → add GROQ_API_KEY to Colab Secrets')

# ─── Initialise Groq Client ──────────────────────────────────────────────────
# Groq uses the same chat completions API structure as OpenAI
client = Groq(api_key=GROQ_API_KEY)

# ─── Console for rich output ─────────────────────────────────────────────────
console = Console()

# ─── Global Configuration ────────────────────────────────────────────────────
CONFIG = {
    'model':         'llama-3.3-70b-versatile',  # Groq's best free model for code
    'temperature':   0.1,              # Low temp = deterministic, consistent code
    # NOTE: Groq is FREE — no cost per token on free tier!
    'max_tokens':    4096,             # Max tokens for C++ output
    'warmup_runs':   3,                # Hyperfine warmup runs (not counted)
    'bench_runs':    10,               # Hyperfine measurement runs
    'output_dir':    Path('/tmp/codexcelerate'),  # Where we save generated files
}

# Create output directory
CONFIG['output_dir'].mkdir(exist_ok=True)

print()
print('─' * 50)
print('⚙️  Configuration:')
for k, v in CONFIG.items():
    print(f'   {k:<15}: {v}')
print('─' * 50)
print('✅ Setup complete. Proceed to Cell 4.')

---
## 📂 Cell 4 — Data Loading: Real Python Programs

### What this cell does:
Defines **3 real, computationally intensive Python programs** that serve as our test dataset.
These are not toy examples — they are real algorithms used in production systems.

### Why these 3 algorithms?
| Algorithm | Why it's slow in Python | Why it's fast in C++ |
|-----------|------------------------|---------------------|
| **Bubble Sort** | O(n²) comparisons × Python's interpreter overhead per swap | Compiled comparisons take nanoseconds; cache-friendly memory access |
| **Prime Sieve (Eratosthenes)** | Array indexing in Python is 50–100× slower than C array access | C++ `vector<bool>` is bit-packed; OpenMP parallelises the inner loop |
| **Matrix Multiplication** | Python loops are slow; NumPy would help but we use pure Python intentionally | C++ with `-march=native` uses SIMD (AVX2) vectorisation automatically |

### ✅ Expected output: Programs printed to screen for inspection

In [ ]:
# ─── CELL 4: Data Loading — Define Python Programs ──────────────────────────
# We define 3 computationally intensive Python programs.
# These will be transpiled to C++, compiled, and benchmarked.

import textwrap

PROGRAMS = {

    # ── Program 1: Bubble Sort ────────────────────────────────────────────────
    # Bubble Sort is O(n2) — intentionally slow for dramatic benchmarking results.
    # 10,000 elements = 100,000,000 comparisons -> Python takes ~800ms, C++ takes <1ms
    'bubble_sort': {
        'description': 'Bubble Sort on 10,000 random integers',
        'complexity':   'O(n^2)',
        'expected_speedup': '500-2000x',
        'code': textwrap.dedent('''
            import random

            # Sort array using bubble sort algorithm
            def bubble_sort(arr):
                n = len(arr)
                for i in range(n):
                    for j in range(0, n - i - 1):
                        if arr[j] > arr[j + 1]:
                            arr[j], arr[j + 1] = arr[j + 1], arr[j]
                return arr

            # Generate 10,000 random integers between 0 and 100,000
            random.seed(42)
            data = [random.randint(0, 100000) for _ in range(10000)]

            result = bubble_sort(data)
            print(f"Sorted {len(result)} elements")
            print(f"First 5: {result[:5]}")
            print(f"Last 5:  {result[-5:]}")
        ''').strip()
    },

    # ── Program 2: Sieve of Eratosthenes ──────────────────────────────────────
    # Classic prime number algorithm — inner loop is highly parallelisable
    # Finding all primes up to 5 million: Python ~2500ms, C++ with OpenMP ~40ms
    'prime_sieve': {
        'description': 'Sieve of Eratosthenes — find all primes up to 5,000,000',
        'complexity':   'O(n log log n)',
        'expected_speedup': '50-200x',
        'code': textwrap.dedent('''
            # Find all prime numbers up to limit using the Sieve of Eratosthenes
            def sieve_of_eratosthenes(limit):
                is_prime = [True] * (limit + 1)
                is_prime[0] = is_prime[1] = False

                i = 2
                while i * i <= limit:
                    if is_prime[i]:
                        for j in range(i * i, limit + 1, i):
                            is_prime[j] = False
                    i += 1

                return [num for num in range(2, limit + 1) if is_prime[num]]

            LIMIT = 5_000_000
            primes = sieve_of_eratosthenes(LIMIT)
            print(f"Found {len(primes)} primes up to {LIMIT:,}")
            print(f"Largest prime: {primes[-1]}")
            print(f"First 10 primes: {primes[:10]}")
        ''').strip()
    },

    # ── Program 3: Matrix Multiplication ──────────────────────────────────────
    # Pure Python matrix multiply — the most extreme speedup example
    # 200x200 matrices: Python ~8000ms, C++ with OpenMP ~2ms -> ~4000x speedup
    'matrix_multiply': {
        'description': 'Matrix multiplication of two 200x200 matrices (pure Python)',
        'complexity':   'O(n^3)',
        'expected_speedup': '1000-8000x',
        'code': textwrap.dedent('''
            import random

            # Create a matrix filled with random floats
            def create_matrix(rows, cols, seed=42):
                random.seed(seed)
                return [[random.uniform(0, 1) for _ in range(cols)] for _ in range(rows)]

            # Multiply two matrices using the standard O(n^3) algorithm
            def matrix_multiply(A, B):
                rows_A = len(A)
                cols_A = len(A[0])
                cols_B = len(B[0])
                C = [[0.0] * cols_B for _ in range(rows_A)]
                for i in range(rows_A):
                    for j in range(cols_B):
                        for k in range(cols_A):
                            C[i][j] += A[i][k] * B[k][j]
                return C

            SIZE = 200
            A = create_matrix(SIZE, SIZE, seed=42)
            B = create_matrix(SIZE, SIZE, seed=123)

            C = matrix_multiply(A, B)
            print(f"Multiplied {SIZE}x{SIZE} matrices")
            print(f"Result[0][0] = {C[0][0]:.6f}")
            print(f"Result[0][1] = {C[0][1]:.6f}")
        ''').strip()
    }
}

# ─── Display Program Summary ──────────────────────────────────────────────────
print('📂 Loaded', len(PROGRAMS), 'Python programs for transpilation:\n')
for name, prog in PROGRAMS.items():
    print(f"  🔹 {name}")
    print(f"     Description:  {prog['description']}")
    print(f"     Complexity:   {prog['complexity']}")
    print(f"     Exp. Speedup: {prog['expected_speedup']}")
    print(f"     Lines of code: {len(prog['code'].splitlines())}")
    print()

print('─' * 50)
print('✅ Programs loaded. Proceed to Cell 5 (EDA).')


---
## 📊 Cell 5 — EDA: Analysing Python Code Characteristics

### What this cell does:
Before transpiling, we perform **Exploratory Data Analysis** on our Python programs.
This is the 'data loading and cleaning' phase adapted for code — we analyse:
- Line count, function count, loop depth
- Estimated complexity
- Which operations are the bottleneck

### Why EDA on code?
In production transpilation systems, code analysis tells you **which programs will benefit most**
from transpilation. CPU-bound (nested loops) → massive speedup. I/O-bound → minimal speedup.
This helps prioritise which code to transpile in a real engineering workflow.

### ✅ Expected output: Plotly charts showing code complexity and loop depth analysis

In [ ]:
# ─── CELL 5: EDA — Analyse Python Program Characteristics ───────────────────
import ast
import re

def analyse_python_code(name, code):
    """
    Static analysis of a Python program.
    Uses ast (Abstract Syntax Tree) for structural analysis.
    Returns a dict of code metrics.
    """
    tree = ast.parse(code)

    # Count different node types
    node_counts = {}
    for node in ast.walk(tree):
        node_type = type(node).__name__
        node_counts[node_type] = node_counts.get(node_type, 0) + 1

    # Measure maximum loop nesting depth
    def max_loop_depth(node, depth=0):
        if isinstance(node, (ast.For, ast.While)):
            depth += 1
        return max([depth] + [max_loop_depth(child, depth)
                              for child in ast.iter_child_nodes(node)])

    lines          = code.splitlines()
    total_lines    = len(lines)
    blank_lines    = sum(1 for l in lines if l.strip() == '')
    comment_lines  = sum(1 for l in lines if l.strip().startswith('#'))
    code_lines     = total_lines - blank_lines - comment_lines

    for_loops      = node_counts.get('For', 0)
    while_loops    = node_counts.get('While', 0)
    functions      = node_counts.get('FunctionDef', 0)
    list_comps     = node_counts.get('ListComp', 0)
    loop_depth     = max_loop_depth(tree)

    # Estimate parallelisability score (0–10)
    # High nested loop depth + no data dependencies = high parallelisability
    parallel_score = min(10, loop_depth * 3 + for_loops)

    return {
        'program':         name,
        'total_lines':     total_lines,
        'code_lines':      code_lines,
        'for_loops':       for_loops,
        'while_loops':     while_loops,
        'functions':       functions,
        'list_comps':      list_comps,
        'max_loop_depth':  loop_depth,
        'parallel_score':  parallel_score,
    }

# ─── Run analysis on all programs ───────────────────────────────────────────
analysis_results = []
for name, prog in PROGRAMS.items():
    metrics = analyse_python_code(name, prog['code'])
    analysis_results.append(metrics)

df = pd.DataFrame(analysis_results)

# ─── Display analysis table ──────────────────────────────────────────────────
print('📊 Code Analysis Results:')
print(df.to_string(index=False))
print()

# ─── Chart 1: Code Lines vs Loop Depth ──────────────────────────────────────
fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Lines of Code per Program',
        'Loop Count (For + While)',
        'Parallelisability Score (0–10)'
    ]
)

colors = ['#3498db', '#e74c3c', '#2ecc71']
programs = df['program'].tolist()

# Subplot 1: Lines of code
fig1.add_trace(
    go.Bar(x=programs, y=df['code_lines'], marker_color=colors,
           text=df['code_lines'], textposition='auto', name='Code Lines'),
    row=1, col=1
)

# Subplot 2: Loop count
fig1.add_trace(
    go.Bar(x=programs, y=df['for_loops'] + df['while_loops'],
           marker_color=colors,
           text=df['for_loops'] + df['while_loops'],
           textposition='auto', name='Loops'),
    row=1, col=2
)

# Subplot 3: Parallelisability score
fig1.add_trace(
    go.Bar(x=programs, y=df['parallel_score'], marker_color=colors,
           text=df['parallel_score'], textposition='auto', name='Parallel Score'),
    row=1, col=3
)

fig1.update_layout(
    title='EDA: Python Code Characteristics Analysis',
    height=400, showlegend=False,
    template='plotly_white',
    font=dict(family='Arial', size=12)
)
fig1.show()

# ─── Chart 2: Max Loop Nesting Depth (key predictor of speedup) ─────────────
fig2 = go.Figure(data=[
    go.Bar(
        x=programs,
        y=df['max_loop_depth'],
        marker_color=['#e74c3c' if d >= 3 else '#f39c12' if d == 2 else '#2ecc71'
                      for d in df['max_loop_depth']],
        text=[f'Depth: {d}' for d in df['max_loop_depth']],
        textposition='auto'
    )
])
fig2.update_layout(
    title='Maximum Loop Nesting Depth — Key Predictor of Speedup Potential',
    xaxis_title='Program', yaxis_title='Loop Nesting Depth',
    annotations=[
        dict(x=0.5, y=1.12, xref='paper', yref='paper', showarrow=False,
             text='🔴 Depth 3+ = Extremely high speedup potential | 🟡 Depth 2 = High | 🟢 Depth 1 = Moderate',
             font=dict(size=11))
    ],
    template='plotly_white', height=350
)
fig2.show()

print()
print('─' * 50)
print('📌 Key EDA Insights:')
print('   • matrix_multiply has depth-3 loops → highest speedup potential')
print('   • bubble_sort has depth-2 loops → very high speedup potential')
print('   • prime_sieve has depth-1 loops but is memory-access heavy')
print('─' * 50)
print('✅ EDA complete. Proceed to Cell 6 (Transpilation).')

---
## 🤖 Cell 6 — Core Pipeline: GPT-4o Two-Pass Transpiler

### What this cell does:
Defines the **heart of CodeXcelerate** — the two-pass GPT-4o transpilation pipeline.

### Why Two Passes?

```
Python Code
    │
    ▼
Pass 1 (GPT-4o) ─── Focus: CORRECTNESS ───► C++ code (correct, but single-threaded)
    │
    ▼
Pass 2 (GPT-4o) ─── Focus: PARALLELISM ──► C++ code (correct + multi-core)
```

If we ask GPT-4o to do both in one pass, it makes mistakes — it tries to add OpenMP 
parallelism while simultaneously translating semantics, and gets confused about data 
dependencies. Separating concerns gives dramatically better results.

### System Prompts Explained:
- **Pass 1 prompt**: Tells GPT-4o to be a C++ expert focused purely on semantic equivalence
- **Pass 2 prompt**: Tells GPT-4o to be an OpenMP expert adding parallelism to already-correct C++

### ⏱️ Expected runtime: 15–30 seconds per program (2 API calls each)
### ✅ Expected output: C++ code displayed for each program

In [ ]:
# ─── CELL 6: Two-Pass GPT-4o Transpiler ─────────────────────────────────────

# ── Pass 1 System Prompt: Focus on CORRECTNESS ───────────────────────────────
# We want semantically equivalent C++ — same algorithm, same output
PASS1_SYSTEM_PROMPT = """
You are a world-class C++17 developer and compiler engineer.

Your task: Convert the given Python code to equivalent, production-grade C++ code.

STRICT RULES:
1. SEMANTIC EQUIVALENCE: The C++ must produce IDENTICAL output to the Python
2. Use modern C++17 features: auto, range-based for, structured bindings
3. Use STL containers: std::vector instead of raw arrays, std::string, unordered_map
4. Include ALL necessary #include statements
5. Include a complete main() function that runs the same computation
6. Handle integer overflow: use long long for large numbers
7. Use fixed-precision random: implement LCG or use <random> with seed 42 where needed
8. Code must compile with: clang++ -O3 -std=c++17

OUTPUT FORMAT: Raw C++ code ONLY. No markdown fences. No explanation. Just the code.
""".strip()

# ── Pass 2 System Prompt: Focus on PARALLELISM ───────────────────────────────
# We take already-correct C++ and add OpenMP multi-core parallelism
PASS2_SYSTEM_PROMPT = """
You are an expert in OpenMP parallel programming and high-performance computing.

Your task: Add OpenMP parallelisation to the given C++ code to maximise performance.

STRICT RULES:
1. Add: #include <omp.h> at the top
2. Add: omp_set_num_threads(omp_get_max_threads()); at start of main()
3. Add #pragma omp parallel for to ALL loops with NO loop-carried dependencies
4. For accumulation loops: use reduction(+:variable) or reduction(max:variable)
5. For single-variable updates in parallel sections: use #pragma omp atomic
6. DO NOT parallelise loops with dependencies (e.g., bubble sort inner swap loop)
7. The parallelised code must produce IDENTICAL results to the sequential version
8. Code must compile with: clang++ -O3 -fopenmp -march=native -std=c++17

OUTPUT FORMAT: Raw C++ code ONLY. No markdown fences. No explanation. Just the code.
""".strip()


def clean_cpp_code(raw_output):
    """
    Remove markdown code fences from GPT-4o output.
    GPT-4o sometimes wraps code in ```cpp ... ``` even when told not to.
    """
    code = raw_output.strip()
    # Remove opening fence (```cpp or ```c++ or ```)
    if code.startswith('```'):
        lines = code.split('\n')
        # Find the closing fence and extract between fences
        start = 1  # Skip first line (the opening fence)
        end = len(lines)
        for i in range(len(lines) - 1, 0, -1):
            if lines[i].strip() == '```':
                end = i
                break
        code = '\n'.join(lines[start:end])
    return code.strip()


def transpile_pass1(python_code, program_name):
    """
    Pass 1: Convert Python to equivalent C++ (no parallelism yet).
    Uses GPT-4o with low temperature for deterministic output.
    """
    print(f'   🔄 Pass 1: Python → C++ (correctness)...')
    start = time.time()

    response = client.chat.completions.create(
        model=CONFIG['model'],
        temperature=CONFIG['temperature'],
        max_tokens=CONFIG['max_tokens'],
        messages=[
            {'role': 'system', 'content': PASS1_SYSTEM_PROMPT},
            {'role': 'user',   'content': f'Convert this Python code to C++:\n\n{python_code}'}
        ]
        # Note: Groq does not require response_format parameter
    )

    cpp_code = clean_cpp_code(response.choices[0].message.content)
    elapsed = time.time() - start
    tokens_used = response.usage.total_tokens

    print(f'   ✅ Pass 1 complete in {elapsed:.1f}s | Tokens used: {tokens_used}')
    return cpp_code, tokens_used


def transpile_pass2(cpp_code, program_name):
    """
    Pass 2: Add OpenMP parallelisation to the C++ code from Pass 1.
    """
    print(f'   🔄 Pass 2: C++ → OpenMP C++ (parallelism)...')
    start = time.time()

    response = client.chat.completions.create(
        model=CONFIG['model'],
        temperature=CONFIG['temperature'],
        max_tokens=CONFIG['max_tokens'],
        messages=[
            {'role': 'system', 'content': PASS2_SYSTEM_PROMPT},
            {'role': 'user',   'content': f'Add OpenMP parallelisation to this C++ code:\n\n{cpp_code}'}
        ]
    )

    cpp_parallel = clean_cpp_code(response.choices[0].message.content)
    elapsed = time.time() - start
    tokens_used = response.usage.total_tokens

    print(f'   ✅ Pass 2 complete in {elapsed:.1f}s | Tokens used: {tokens_used}')
    return cpp_parallel, tokens_used


def full_transpile(program_name, python_code):
    """
    Full two-pass transpilation pipeline.
    Returns dict with both C++ versions and metadata.
    """
    print(f'\n⚡ Transpiling: {program_name}')
    print('   ' + '─' * 44)

    cpp_basic,    tokens1 = transpile_pass1(python_code, program_name)
    cpp_parallel, tokens2 = transpile_pass2(cpp_basic,   program_name)

    print(f'   📊 Total tokens: {tokens1 + tokens2} | Cost: $0.00 (Groq free tier 🎉)')

    return {
        'name':         program_name,
        'python':       python_code,
        'cpp_basic':    cpp_basic,
        'cpp_parallel': cpp_parallel,
        'total_tokens': tokens1 + tokens2
    }


# ─── Run transpilation for all programs ─────────────────────────────────────
print('🚀 Starting CodeXcelerate transpilation pipeline...')
print('='*52)

transpilation_results = {}
for name, prog in PROGRAMS.items():
    result = full_transpile(name, prog['code'])
    transpilation_results[name] = result
    # Brief pause to avoid rate limiting
    time.sleep(1)

print()
print('='*52)
print('✅ Transpilation complete for all', len(PROGRAMS), 'programs!')
print()

# Preview the generated C++ for bubble_sort
sample = transpilation_results['bubble_sort']
print('📄 Preview: bubble_sort C++ (first 20 lines):')
print('─' * 52)
lines = sample['cpp_parallel'].splitlines()[:20]
for i, line in enumerate(lines, 1):
    print(f'  {i:2}│ {line}')
print('  ...(truncated for display)')
print('─' * 52)
print('✅ Proceed to Cell 7 (Compilation).')

---
## 🔨 Cell 7 — Compilation: clang++ with -O3 and OpenMP

### What this cell does:
Compiles each GPT-4o-generated C++ program using `clang++` with aggressive optimisation flags.

### Compiler Flags Explained:
| Flag | Meaning | Effect |
|------|---------|--------|
| `-O3` | Optimisation level 3 (maximum) | Auto-vectorisation, loop unrolling, inlining |
| `-fopenmp` | Enable OpenMP | Activates `#pragma omp` directives |
| `-march=native` | Optimise for THIS specific CPU | Enables AVX2, SSE4.2 SIMD instructions |
| `-std=c++17` | Use C++17 standard | Access to modern features |
| `-o output` | Output binary name | Where the compiled executable goes |

### What happens if compilation fails?
This cell includes **automatic error detection and retry logic**. If GPT-4o generated 
code with a compilation error, we capture the error message and attempt a fix using 
one more GPT-4o call with the error as context.

### ✅ Expected output: 'Compiled successfully' for all 3 programs

In [ ]:
# ─── CELL 7: Compilation with Error Recovery ────────────────────────────────
# SAFETY CHECK: Verify clang++ is available before attempting compilation
# If this fails, go back and re-run Cell 1 first!

import shutil, subprocess as _sp

if not shutil.which('clang++'):
    print('⚠️  clang++ not found! Installing now...')
    _sp.run(['apt-get', 'install', '-y', 'clang', 'libomp-dev'], capture_output=True)
    if shutil.which('clang++'):
        print('✅ clang++ installed successfully. Continuing...')
    else:
        raise EnvironmentError(
            'clang++ could not be installed. '
            'Please re-run Cell 1 manually, then re-run this cell.'
        )
else:
    print('✅ clang++ found:', shutil.which('clang++'))

print()


def compile_cpp(cpp_code, program_name, output_dir=None):
    """
    Compiles C++ code using clang++ with maximum optimisation.
    Saves source to /tmp, compiles, returns path to binary.
    """
    if output_dir is None:
        output_dir = CONFIG['output_dir']

    src_path = output_dir / f'{program_name}.cpp'
    bin_path = output_dir / program_name

    # Write C++ source code to file
    src_path.write_text(cpp_code, encoding='utf-8')

    # Build compile command
    # -O3: Maximum compiler optimisation
    # -fopenmp: Enable OpenMP multi-threading
    # -march=native: Use SIMD instructions for THIS CPU
    # -std=c++17: Modern C++ standard
    # Use clang++ if symlink exists, else fall back to clang++-14
    # Find compiler: use path stored by Cell 1, or search automatically
    import shutil as _sh, glob as _glob, os as _os
    _clangpp = (
        _os.environ.get('CLANGPP_PATH') or
        _sh.which('clang++') or
        next(iter(sorted(_glob.glob('/usr/bin/clang++*'), reverse=True)), None)
    )
    if not _clangpp:
        raise EnvironmentError('clang++ not found — re-run Cell 1 first!')
    compiler = _clangpp

    cmd = [
        compiler,
        '-O3',
        '-fopenmp',
        '-march=native',
        '-std=c++17',
        str(src_path),
        '-o', str(bin_path)
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=60
    )

    return {
        'success':    result.returncode == 0,
        'binary':     str(bin_path),
        'source':     str(src_path),
        'stderr':     result.stderr,
        'stdout':     result.stdout,
        'returncode': result.returncode
    }


def fix_compilation_error(cpp_code, error_message):
    """
    If compilation fails, ask GPT-4o to fix the specific error.
    This is the 'self-healing' feature of CodeXcelerate.
    Uses Groq LLM to read the compiler error and fix the code.
    """
    print('   🔧 Attempting auto-fix with Groq LLM...')

    fix_prompt = f"""
This C++ code has a compilation error. Fix it.

COMPILATION ERROR:
{error_message[:1000]}  

BROKEN C++ CODE:
{cpp_code}

Return ONLY the fixed C++ code with no explanation and no markdown fences.
    """.strip()

    response = client.chat.completions.create(
        model=CONFIG['model'],
        temperature=0.1,
        max_tokens=CONFIG['max_tokens'],
        messages=[{'role': 'user', 'content': fix_prompt}]
    )
    return clean_cpp_code(response.choices[0].message.content)


def compile_with_retry(cpp_code, program_name, max_retries=2):
    """
    Try to compile. If it fails, ask GPT-4o to fix and retry.
    """
    for attempt in range(max_retries + 1):
        result = compile_cpp(cpp_code, program_name)

        if result['success']:
            return result, cpp_code

        print(f'   ⚠️  Attempt {attempt+1} failed:')
        # Show first 3 lines of error for diagnosis
        error_lines = result['stderr'].strip().split('\n')[:3]
        for line in error_lines:
            print(f'      {line}')

        if attempt < max_retries:
            cpp_code = fix_compilation_error(cpp_code, result['stderr'])
        else:
            print(f'   ❌ Could not fix after {max_retries} attempts')

    return result, cpp_code


# ─── Also save Python programs to files for benchmarking ────────────────────
def save_python_program(name, code):
    py_path = CONFIG['output_dir'] / f'{name}.py'
    py_path.write_text(code, encoding='utf-8')
    return str(py_path)


# ─── Run compilation for all programs ────────────────────────────────────────
print('🔨 Compiling all transpiled C++ programs...')
print('='*52)
print(f'   Flags: clang++ -O3 -fopenmp -march=native -std=c++17')
print()

compilation_results = {}
for name, trans in transpilation_results.items():
    print(f'⚙️  Compiling: {name}')

    # Compile the OpenMP-parallelised version
    comp_result, final_cpp = compile_with_retry(
        trans['cpp_parallel'], name
    )

    # Save Python file for benchmarking
    py_path = save_python_program(name, trans['python'])

    compilation_results[name] = {
        **comp_result,
        'py_path':   py_path,
        'final_cpp': final_cpp
    }

    if comp_result['success']:
        # Quick correctness check — run the binary
        test = subprocess.run(
            [comp_result['binary']], capture_output=True, text=True, timeout=30
        )
        print(f'   ✅ Compiled! Binary: {comp_result["binary"]}')
        print(f'   🔹 Test output: {test.stdout.strip()[:80]}')
    else:
        print(f'   ❌ Compilation failed for {name}')
    print()

# Summary
success_count = sum(1 for r in compilation_results.values() if r['success'])
print('='*52)
print(f'📊 Compilation results: {success_count}/{len(PROGRAMS)} successful')
print('✅ Proceed to Cell 8 (Benchmarking).')

---
## ⏱️ Cell 8 — Benchmarking: Hyperfine Statistical Analysis

### What this cell does:
Runs **rigorous statistical benchmarks** comparing Python vs C++ execution time.

### Why Hyperfine instead of Python's `time` module?
| Method | Problem |
|--------|---------|
| `time python script.py` | Single measurement — unreliable due to OS scheduling noise |
| Python `timeit` | Measures Python interpreter startup + run; can't benchmark binary |
| **Hyperfine** | Runs program N times, computes mean, stddev, min, max, removes outliers |

### What Hyperfine does:
1. **Warmup runs** (3): Let OS cache files, CPU warm up — not counted
2. **Measurement runs** (10): Timed runs — compute statistics
3. **Output**: Mean ± stddev, min, max in JSON format

### Statistical validity:
10 runs gives us ~95% confidence interval on the mean execution time.
This is the same methodology used in academic CS performance papers.

### ⏱️ Expected runtime: 3–8 minutes (each Python run takes 1–30 seconds)
### ✅ Expected output: Speedup table showing 100×–10,000× improvements

In [ ]:
# ─── CELL 8: Hyperfine Benchmarking ─────────────────────────────────────────

def run_hyperfine_benchmark(py_path, cpp_binary, program_name,
                             warmup=3, runs=10):
    """
    Runs Hyperfine to benchmark Python vs C++ binary.
    Exports results as JSON for parsing.

    Returns dict with timing statistics.
    """
    json_output = CONFIG['output_dir'] / f'{program_name}_bench.json'

    # Hyperfine command:
    # --warmup N    : run N times before measuring (cache warmup)
    # --runs N      : number of timed runs
    # --export-json : save structured results to JSON
    # Two commands: Python script vs C++ binary
    cmd = [
        'hyperfine',
        '--warmup',      str(warmup),
        '--runs',        str(runs),
        '--export-json', str(json_output),
        '--style',       'none',       # No ANSI colours (clean for notebook)
        f'python3 {py_path}',          # Command 1: Python
        cpp_binary                     # Command 2: C++ binary
    ]

    print(f'   📏 Benchmarking {program_name}...')
    print(f'      Python: python3 {py_path}')
    print(f'      C++:    {cpp_binary}')
    print(f'      Config: {warmup} warmup + {runs} measurement runs')

    start_time = time.time()
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=600   # 10 minute timeout for slow Python programs
    )
    elapsed = time.time() - start_time

    if result.returncode != 0:
        print(f'   ❌ Hyperfine error: {result.stderr[:200]}')
        return None

    # Parse JSON output
    with open(json_output) as f:
        data = json.load(f)

    # results[0] = Python, results[1] = C++
    py_result  = data['results'][0]
    cpp_result = data['results'][1]

    # Convert seconds to milliseconds
    py_mean_ms  = py_result['mean'] * 1000
    py_std_ms   = py_result['stddev'] * 1000
    py_min_ms   = py_result['min'] * 1000
    py_max_ms   = py_result['max'] * 1000

    cpp_mean_ms = cpp_result['mean'] * 1000
    cpp_std_ms  = cpp_result['stddev'] * 1000
    cpp_min_ms  = cpp_result['min'] * 1000
    cpp_max_ms  = cpp_result['max'] * 1000

    speedup = py_mean_ms / max(cpp_mean_ms, 0.001)  # Avoid div by zero

    bench_data = {
        'program':         program_name,
        'py_mean_ms':      round(py_mean_ms, 2),
        'py_std_ms':       round(py_std_ms, 2),
        'py_min_ms':       round(py_min_ms, 2),
        'py_max_ms':       round(py_max_ms, 2),
        'cpp_mean_ms':     round(cpp_mean_ms, 4),
        'cpp_std_ms':      round(cpp_std_ms, 4),
        'cpp_min_ms':      round(cpp_min_ms, 4),
        'cpp_max_ms':      round(cpp_max_ms, 4),
        'speedup':         round(speedup, 1),
        'bench_time_s':    round(elapsed, 1),
        'runs':            runs
    }

    print(f'   ✅ Done in {elapsed:.0f}s')
    print(f'      Python: {py_mean_ms:.1f} ms ± {py_std_ms:.1f}')
    print(f'      C++:    {cpp_mean_ms:.3f} ms ± {cpp_std_ms:.3f}')
    print(f'      🚀 Speedup: {speedup:.0f}×')

    return bench_data


# ─── Run benchmarks ──────────────────────────────────────────────────────────
print('⏱️  Starting Hyperfine benchmarks...')
print('='*52)
print('⚠️  Note: Python programs are slow by design — be patient!')
print()

benchmark_results = []

for name, comp in compilation_results.items():
    if not comp['success']:
        print(f'⏭️  Skipping {name} (compilation failed)')
        continue

    bench = run_hyperfine_benchmark(
        py_path=comp['py_path'],
        cpp_binary=comp['binary'],
        program_name=name,
        warmup=CONFIG['warmup_runs'],
        runs=CONFIG['bench_runs']
    )

    if bench:
        benchmark_results.append(bench)
    print()

# Create DataFrame for easier analysis
bench_df = pd.DataFrame(benchmark_results)

print('='*52)
print('📊 BENCHMARK SUMMARY TABLE')
print('='*52)
print()
if len(bench_df) > 0:
    summary_cols = ['program', 'py_mean_ms', 'cpp_mean_ms', 'speedup']
    print(bench_df[summary_cols].to_string(index=False))
print()
print('✅ Benchmarking complete. Proceed to Cell 9 (Evaluation).')

---
## 📈 Cell 9 — Evaluation: Deep Analysis and Visualisation

### What this cell does:
Performs **multi-dimensional evaluation** of our transpilation results:

1. **Speedup Analysis**: How much faster is C++ vs Python for each program?
2. **Statistical Reliability**: Mean ± stddev, min, max — how consistent is the speedup?
3. **Cost Analysis**: What did each GPT-4o call cost?
4. **ROI Analysis**: Tokens spent per speedup achieved

### What makes a good speedup?
- **1–10×**: CPU-bound Python with some overhead → good but not impressive
- **10–100×**: Nested loops with compiler optimisation → very good
- **100–1000×**: Python interpreter overhead + compiler vectorisation → excellent
- **1000–60,000×**: Pure computation loops + OpenMP parallelism → extraordinary

### ✅ Expected output: 4 interactive Plotly charts + evaluation summary table

In [ ]:
# ─── CELL 9: Evaluation and Visualisation ───────────────────────────────────

if len(bench_df) == 0:
    print('⚠️  No benchmark results to visualise. Check Cell 8 for errors.')
else:
    colors_python = '#3498db'
    colors_cpp    = '#e74c3c'
    program_names = bench_df['program'].tolist()

    # ── Chart 1: Main Speedup Chart (log scale) ──────────────────────────────
    fig1 = go.Figure()

    fig1.add_trace(go.Bar(
        name='Python',
        x=program_names,
        y=bench_df['py_mean_ms'],
        error_y=dict(type='data', array=bench_df['py_std_ms'].tolist(), visible=True),
        marker_color=colors_python,
        text=[f'{v:.1f}ms' for v in bench_df['py_mean_ms']],
        textposition='auto'
    ))

    fig1.add_trace(go.Bar(
        name='C++ (OpenMP)',
        x=program_names,
        y=bench_df['cpp_mean_ms'],
        error_y=dict(type='data', array=bench_df['cpp_std_ms'].tolist(), visible=True),
        marker_color=colors_cpp,
        text=[f'{v:.3f}ms' for v in bench_df['cpp_mean_ms']],
        textposition='auto'
    ))

    fig1.update_layout(
        title='⚡ CodeXcelerate: Python vs C++ Execution Time (Log Scale)',
        xaxis_title='Program',
        yaxis_title='Execution Time (ms) — Log Scale',
        yaxis_type='log',
        barmode='group',
        template='plotly_white',
        height=450,
        legend=dict(x=0.75, y=0.95),
        annotations=[dict(
            x=0.5, y=1.08, xref='paper', yref='paper', showarrow=False,
            text='Error bars show ± 1 standard deviation across 10 runs (statistical reliability)',
            font=dict(size=11, color='#666')
        )]
    )
    fig1.show()

    # ── Chart 2: Speedup Factor (the hero chart) ─────────────────────────────
    speedup_colors = [
        '#e74c3c' if s > 1000 else
        '#e67e22' if s > 100 else
        '#f1c40f' if s > 10 else '#2ecc71'
        for s in bench_df['speedup']
    ]

    fig2 = go.Figure(data=[
        go.Bar(
            x=program_names,
            y=bench_df['speedup'],
            marker_color=speedup_colors,
            text=[f'{s:.0f}×' for s in bench_df['speedup']],
            textposition='outside',
            textfont=dict(size=16, color='black')
        )
    ])

    fig2.update_layout(
        title='🚀 Speedup Factor: C++ vs Python (Higher = Better)',
        xaxis_title='Program',
        yaxis_title='Speedup Factor (×)',
        yaxis_type='log',
        template='plotly_white',
        height=400,
        annotations=[
            dict(x=0.5, y=1.08, xref='paper', yref='paper', showarrow=False,
                 text='🔴 >1000× | 🟠 >100× | 🟡 >10× | 🟢 >1×',
                 font=dict(size=11))
        ]
    )
    fig2.show()

    # ── Chart 3: Timing Distribution (min / mean / max) ──────────────────────
    fig3 = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Python Timing Distribution (ms)', 'C++ Timing Distribution (ms)']
    )

    for prog in program_names:
        row_data = bench_df[bench_df['program'] == prog].iloc[0]

        # Python: min-mean-max
        fig3.add_trace(go.Scatter(
            x=[prog, prog, prog],
            y=[row_data['py_min_ms'], row_data['py_mean_ms'], row_data['py_max_ms']],
            mode='markers+lines', name=f'{prog} (Python)',
            marker=dict(size=[8, 14, 8], color=colors_python),
            showlegend=(prog == program_names[0])
        ), row=1, col=1)

        # C++: min-mean-max
        fig3.add_trace(go.Scatter(
            x=[prog, prog, prog],
            y=[row_data['cpp_min_ms'], row_data['cpp_mean_ms'], row_data['cpp_max_ms']],
            mode='markers+lines', name=f'{prog} (C++)',
            marker=dict(size=[8, 14, 8], color=colors_cpp),
            showlegend=(prog == program_names[0])
        ), row=1, col=2)

    fig3.update_layout(
        title='Timing Distribution: Min / Mean / Max across 10 Runs',
        template='plotly_white', height=350
    )
    fig3.show()

    # ── Chart 4: Tokens used vs Speedup achieved (ROI) ───────────────────────
    tokens_used = [
        transpilation_results[name]['total_tokens']
        for name in program_names
        if name in transpilation_results
    ]

    if tokens_used:
        costs = [0.0 for t in tokens_used]  # Groq free tier = $0.00 cost!

        fig4 = make_subplots(rows=1, cols=2,
                             subplot_titles=['API Tokens Used per Program',
                                            'Estimated API Cost (USD)'])

        fig4.add_trace(go.Bar(
            x=program_names[:len(tokens_used)], y=tokens_used,
            marker_color='#9b59b6',
            text=[f'{t:,}' for t in tokens_used], textposition='auto'
        ), row=1, col=1)

        fig4.add_trace(go.Bar(
            x=program_names[:len(costs)], y=costs,
            marker_color='#1abc9c',
            text=[f'${c:.4f}' for c in costs], textposition='auto'
        ), row=1, col=2)

        fig4.update_layout(
            title='API Usage & Cost Analysis (GPT-4o, 2 passes per program)',
            template='plotly_white', height=350, showlegend=False
        )
        fig4.show()

    # ── Final Evaluation Table ────────────────────────────────────────────────
    print()
    print('='*65)
    print('📊 FINAL EVALUATION SUMMARY')
    print('='*65)
    print(f'{"Program":<20} {"Python (ms)":>12} {"C++ (ms)":>10} {"Speedup":>10} {"Grade":>8}')
    print('─'*65)
    for _, row in bench_df.iterrows():
        s = row['speedup']
        grade = ('🔴 EXTREME' if s > 1000 else
                 '🟠 EXCELLENT' if s > 100 else
                 '🟡 VERY GOOD' if s > 10 else '🟢 GOOD')
        print(f'{row["program"]:<20} {row["py_mean_ms"]:>12.1f} {row["cpp_mean_ms"]:>10.3f} {s:>9.0f}× {grade}')
    print('='*65)
    print(f'Average speedup: {bench_df["speedup"].mean():.0f}×')
    print(f'Maximum speedup: {bench_df["speedup"].max():.0f}×')
    print()
    print('✅ Evaluation complete. Proceed to Cell 10 (Final Outputs).')

---
## 💾 Cell 10 — Final Outputs: Save Results and Generate Report

### What this cell does:
Saves all results in multiple formats for submission and future reference:

1. **CSV file** — Benchmark results table (for Excel/Sheets analysis)
2. **JSON file** — Complete results including all C++ code (for programmatic access)
3. **HTML report** — Standalone viewable report with embedded charts
4. **Colab download** — Download all files directly from the notebook

### For your GitHub submission:
The JSON output contains all transpiled C++ code — you can extract it and add to your repo.

### ✅ Expected output: Download links for all output files

In [ ]:
# ─── CELL 10: Save Final Outputs ─────────────────────────────────────────────
from google.colab import files
import io

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = CONFIG['output_dir']

# ── 1. Save benchmark CSV ────────────────────────────────────────────────────
csv_path = output_dir / f'benchmark_results_{timestamp}.csv'
bench_df.to_csv(csv_path, index=False)
print(f'💾 Saved benchmark CSV: {csv_path}')

# ── 2. Save complete JSON results ────────────────────────────────────────────
json_results = {
    'timestamp': timestamp,
    'model_used': 'Groq: ' + CONFIG['model'],
    'api_provider': 'Groq (Free Tier)',
    'cost_usd': 0.00,
    'config': {k: str(v) for k, v in CONFIG.items()},
    'programs': {}
}

for name in transpilation_results:
    trans = transpilation_results[name]
    bench_row = bench_df[bench_df['program'] == name]

    json_results['programs'][name] = {
        'description': PROGRAMS[name]['description'],
        'python_code': trans['python'],
        'cpp_basic':   trans['cpp_basic'],
        'cpp_openmp':  trans['cpp_parallel'],
        'tokens_used': trans['total_tokens'],
        'benchmark': bench_row.to_dict('records')[0] if len(bench_row) > 0 else None
    }

json_path = output_dir / f'codexcelerate_results_{timestamp}.json'
with open(json_path, 'w') as f:
    json.dump(json_results, f, indent=2)
print(f'💾 Saved JSON results: {json_path}')

# ── 3. Generate HTML Report ──────────────────────────────────────────────────
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>CodeXcelerate — Day 30 Milestone Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; max-width: 900px; margin: 40px auto; padding: 20px; }}
        h1   {{ color: #1a5e38; }}
        h2   {{ color: #27ae60; border-bottom: 2px solid #27ae60; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
        th    {{ background: #1a5e38; color: white; padding: 10px; }}
        td    {{ border: 1px solid #ddd; padding: 8px; }}
        tr:nth-child(even) {{ background: #f9f9f9; }}
        .speedup {{ font-weight: bold; color: #e74c3c; }}
        pre  {{ background: #2b2b2b; color: #d4d4d4; padding: 15px; border-radius: 5px; overflow-x: auto; }}
    </style>
</head>
<body>
    <h1>⚡ CodeXcelerate: Python-to-C++ AI Transpiler</h1>
    <p><strong>AlgoProfessor AI R&D Internship</strong> | Day 30 | Milestone 5</p>
    <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>

    <h2>📊 Benchmark Results</h2>
    <table>
        <tr><th>Program</th><th>Python (ms)</th><th>C++ (ms)</th><th>Speedup</th><th>Grade</th></tr>
"""

for _, row in bench_df.iterrows():
    s = row['speedup']
    grade = '🔴 EXTREME' if s > 1000 else '🟠 EXCELLENT' if s > 100 else '🟡 VERY GOOD' if s > 10 else '🟢 GOOD'
    html_content += f"""
        <tr>
            <td>{row['program']}</td>
            <td>{row['py_mean_ms']:.1f} ± {row['py_std_ms']:.1f}</td>
            <td>{row['cpp_mean_ms']:.3f} ± {row['cpp_std_ms']:.3f}</td>
            <td class='speedup'>{s:.0f}×</td>
            <td>{grade}</td>
        </tr>"""

html_content += """
    </table>

    <h2>🛠️ Technology Stack</h2>
    <ul>
        <li><strong>GPT-4o</strong> — Two-pass AI transpilation (Pass 1: Correctness, Pass 2: OpenMP)</li>
        <li><strong>Clang++</strong> — C++ compilation with -O3 -fopenmp -march=native</li>
        <li><strong>OpenMP</strong> — CPU parallelism (#pragma omp parallel for)</li>
        <li><strong>Hyperfine</strong> — Statistical benchmarking (10 runs + 3 warmup)</li>
        <li><strong>Plotly</strong> — Interactive visualisation</li>
    </ul>

    <h2>📝 Generated C++ Code Samples</h2>
"""

for name, trans in transpilation_results.items():
    preview = '\n'.join(trans['cpp_parallel'].splitlines()[:25])
    html_content += f"""
    <h3>{name} (first 25 lines)</h3>
    <pre>{preview}\n    ...(truncated)</pre>
    """

html_content += "</body></html>"

html_path = output_dir / f'codexcelerate_report_{timestamp}.html'
html_path.write_text(html_content)
print(f'💾 Saved HTML report: {html_path}')

# ── 4. Download all files ────────────────────────────────────────────────────
print()
print('📥 Downloading files to your computer...')
print()

try:
    print('   Downloading benchmark CSV...')
    files.download(str(csv_path))

    print('   Downloading JSON results...')
    files.download(str(json_path))

    print('   Downloading HTML report...')
    files.download(str(html_path))
except Exception as e:
    print(f'   Note: Auto-download may require manual approval: {e}')
    print(f'   Files are at: {output_dir}')

print()
print('='*52)
print('🎉 CodeXcelerate Milestone 5 COMPLETE!')
print()
print('📁 Output files saved:')
print(f'   • {csv_path.name}  — Benchmark results')
print(f'   • {json_path.name} — Full results + C++ code')
print(f'   • {html_path.name} — HTML report')
print()
print('📌 Next steps:')
print('   1. File → Download → Download .ipynb to save this notebook')
print('   2. Upload notebook + CSV + JSON to your GitHub repo')
print('   3. Add HTML report to your outputs/ folder')
print('   4. Commit: [Day-30] Milestone 5 - CodeXcelerate Python-to-C++ Transpiler')
print('='*52)

---
# 🔧 Step-by-Step Execution Guide
## How to Run This Notebook in Google Colab

---

## STEP 1: Open Google Colab
1. Go to **colab.research.google.com**
2. Sign in with your Google account
3. You get **free GPU/CPU access** — no payment required for this project

---

## STEP 2: Upload This Notebook
**Option A** (Recommended): Upload the .ipynb file
1. Click **File → Upload notebook**
2. Select `CodeXcelerate_Day30_Milestone5.ipynb` from your computer
3. The notebook opens automatically

**Option B**: Open directly from GitHub
1. Click **File → Open notebook → GitHub tab**
2. Paste your GitHub repo URL
3. Select the .ipynb file

---

## STEP 3: Get Your FREE Groq API Key and Add to Colab

**Getting the key (takes 2 minutes):**
1. Go to **console.groq.com**
2. Click **Sign Up** (free — no credit card required)
3. Click **"API Keys"** in the left sidebar
4. Click **"Create API Key"** → give it a name → copy the key

**Adding to Colab Secrets:**
1. Click the **🔑 key icon** in the left sidebar of Colab
2. Click **"Add new secret"**
3. Name: `GROQ_API_KEY`
4. Value: paste your Groq key (starts with `gsk_...`)
5. Toggle **"Notebook access"** to ON ← This is critical!
6. 💚 **Cost: $0.00 — Groq is completely free!**

---

## STEP 4: Run Cells in Order
⚠️ **IMPORTANT: Run cells top to bottom. Never skip a cell.**

| Cell | What it does | Time |
|------|-------------|------|
| Cell 1 | Install clang, hyperfine | 1–3 min |
| Cell 2 | Install Python libraries | 30–60 sec |
| Cell 3 | Configure API | <5 sec |
| Cell 4 | Load Python programs | <5 sec |
| Cell 5 | EDA + charts | <10 sec |
| Cell 6 | GPT-4o transpilation (3 programs × 2 passes) | 2–5 min |
| Cell 7 | Compile all C++ | 30–60 sec |
| Cell 8 | Benchmark all programs | **5–15 min** |
| Cell 9 | Evaluation + charts | <15 sec |
| Cell 10 | Save + download outputs | <30 sec |

**Total expected runtime: 10–25 minutes**

To run all cells at once: **Runtime → Run all** (Ctrl+F9)
To run a single cell: Click it, press **Shift+Enter**

---

## STEP 5: Download Your Notebook
1. **File → Download → Download .ipynb**
2. This saves the notebook with all outputs to your computer
3. The file is typically 2–5 MB with all charts embedded

---

## STEP 6: Upload to GitHub
```bash
# In your terminal (after downloading):
cd algoprofessor-rd-internship-2026/
mkdir -p day30_milestone5/
cp ~/Downloads/CodeXcelerate_Day30_Milestone5.ipynb day30_milestone5/
cp ~/Downloads/benchmark_results_*.csv day30_milestone5/outputs/
cp ~/Downloads/codexcelerate_results_*.json day30_milestone5/outputs/
git add day30_milestone5/
git commit -m "[Day-30] Milestone 5 - CodeXcelerate Python-to-C++ AI Transpiler"
git push origin main
```

---

## ⚠️ Common Errors and Fixes

| Error | Cause | Fix |
|-------|-------|-----|
| `hyperfine: command not found` | Colab session restarted | Re-run Cell 1 |
| `clang++: error: unsupported option '-fopenmp'` | libomp-dev not installed | Run: `!apt-get install -y libomp-dev` |
| `AuthenticationError: Invalid API Key` | API key not set | Check Secrets (🔑 icon) — ensure toggle is ON |
| `RateLimitError` | Too many GPT-4o calls | Wait 60 seconds, add `time.sleep(3)` between transpile calls |
| Compilation error: `expected ';'` | GPT-4o included markdown fences | `clean_cpp_code()` handles this — re-run Cell 6 |
| Benchmark timeout (>10 min) | matrix_multiply too large | Reduce SIZE from 200 to 100 in Cell 4 |
| `Google Drive quota exceeded` | Too many downloads | Download files one at a time manually from `/tmp/codexcelerate/` |
| No speedup (1×) | C++ binary not found | Check Cell 7 for compilation errors first |

---

## 📌 Pro Tips
- **Enable GPU runtime**: Runtime → Change runtime type → T4 GPU (free). OpenMP uses CPU, but it won't hurt.
- **If Colab disconnects**: All `/tmp/` files are lost. Re-run from Cell 1. Outputs in Cells 9–10 were saved to your computer.
- **Check your Groq usage**: console.groq.com → Usage dashboard (free tier resets daily).
- **For your viva**: Know that `-O3` enables auto-vectorisation, and `-march=native` enables SIMD — these alone give 5–50× speedup before OpenMP even helps.

---
# 🎓 Viva Quick Reference

Keep this open during your viva presentation.

## Q: Why does Python's bubble sort run ~1000× slower than C++?
**A:** Three compounding factors:
1. Python interpreter overhead: every `arr[j] > arr[j+1]` is a bytecode instruction with type checking
2. Python objects: even integers are full Python objects with reference counts (28 bytes vs 8 bytes in C++)
3. C++ with `-O3 -march=native` auto-vectorises the comparison loop using SSE/AVX SIMD instructions

## Q: What does OpenMP actually do?
**A:** OpenMP tells the compiler to split a `for` loop across multiple CPU threads automatically.
`#pragma omp parallel for` before a loop → each iteration is assigned to a different CPU core.
On a 4-core machine: theoretical 4× additional speedup on top of sequential C++ optimisations.

## Q: Why two GPT-4o passes?
**A:** Separation of concerns. Pass 1 focuses on semantic correctness — matching Python's algorithm exactly.
Pass 2 focuses on parallelism — identifying which loops have no data dependencies and can run in parallel.
Combining both objectives in one pass leads to errors (incorrect parallelisation of dependent loops).

## Q: How does Hyperfine ensure reliable measurements?
**A:** Warmup runs prime the OS page cache and CPU branch predictor. Multiple runs account for
OS scheduling noise. Hyperfine reports mean ± stddev, letting you quantify measurement reliability.

## Q: What is the architectural bottleneck you cannot solve with this approach?
**A:** Semantic verification. GPT-4o cannot *prove* the C++ is equivalent to the Python for all inputs —
it can only generate code that looks equivalent. For production use, you need a test suite that
validates output equivalence across a range of inputs before deployment. This is the unsolved problem
in AI-assisted code generation.

---
*AlgoProfessor AI R&D Internship | Day 30 | Milestone 5 — CodeXcelerate*
*ceo@algoprofessor.com | www.algoprofessor.com*